In [2]:
import pandas as pd

# Phase 1

In [3]:
df = pd.read_csv("fake_job_postings.csv")
# df.head()
# df.info()
df['fraudulent'].value_counts()

fraudulent
0    17014
1      866
Name: count, dtype: int64

In [4]:
df.columns

Index(['job_id', 'title', 'location', 'department', 'salary_range',
       'company_profile', 'description', 'requirements', 'benefits',
       'telecommuting', 'has_company_logo', 'has_questions', 'employment_type',
       'required_experience', 'required_education', 'industry', 'function',
       'fraudulent'],
      dtype='object')

In [5]:
columns_needed = [
    'title',
    'company_profile',
    'description',
    'requirements',
    'benefits',
    'employment_type',
    'required_experience',
    'required_education',
    'fraudulent'
]
df=df[columns_needed]

# Phase 2

In [6]:
text_columns = ['title','company_profile','description','requirements','benefits']

for col in text_columns:
    df[col]=df[col].fillna('')

for col in text_columns:
    df[col]=df[col].str.lower()

In [ ]:
categorical_columns = ['employment_type','required_experience','required_education']

for col in categorical_columns:
    df[col]=df[col].fillna('Unknown')

In [8]:
df = df.drop_duplicates()


In [9]:
df.isna().sum()

title                  0
company_profile        0
description            0
requirements           0
benefits               0
employment_type        0
required_experience    0
required_education     0
fraudulent             0
dtype: int64

# Phase 3

In [10]:
description_length=[]
for text in df['description']:
    length=len(text)
    description_length.append(length)

df['desciption_length']=description_length

In [ ]:
requirements_length=[]
for text in df['requirements']:
    length = len(text)
    requirements_length.append(length)

df['requirements_length']=requirements_length




In [12]:
benefits_length = []
for text in df['benefits']:
    length = len(text)
    benefits_length.append(length)

df['benefits_length']=benefits_length

In [13]:
description_word_count=[]
for text in df['description']:
    words = text.split()
    count = len(words)
    description_word_count.append(count)

df['description_word_count']=description_word_count

In [14]:
suspicious_words = [
    "work from home",
    "easy money",
    "quick cash",
    "earn per day",
    "earn daily",
    "guaranteed income",
    "no experience needed",
    "no skills required",
    "anyone can apply",
    "simple typing job",
    "data entry work",
    "form filling job",
    "online job",
    "part time job",
    "just few hours work",
    "no interview required",
    "limited slots",
    "urgent hiring",
    "apply immediately",
    "last chance",
    "registration fee",
    "joining fee",
    "training fee",
    "processing fee",
    "security deposit",
    "pay to start",
    "contact on whatsapp",
    "telegram only",
    "dm for details",
    "personal number",
    "government approved",
    "iso certified",
    "verified job",
    "instant payment",
    "weekly payout",
    "paid instantly"
]


In [15]:
suspicious_flag = []

for text in df['description']:
    found=0
    for word in suspicious_words:
        if word in text:
            found=1
            break
    suspicious_flag.append(found)

df['suspicious_flag']=suspicious_flag

In [16]:
company_profile_missing =[]
for text in df['company_profile']:
    if text.split()=='':
     company_profile_missing.append(1)
    else:
        company_profile_missing.append(0)

df['company_profile_missing']=company_profile_missing


# PHASE 4

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [18]:
encoder = OneHotEncoder(handle_unknown='ignore',sparse_output=False)

# encoded_features_names = encoder.get_feature_names_out(categorical_columns)
encoded_array = encoder.fit_transform(df[categorical_columns])
encoded_df=pd.DataFrame(encoded_array,columns=encoder.get_feature_names_out(categorical_columns),index=df.index)

df = pd.concat([df,encoded_df],axis=1)

df= df.drop(columns=categorical_columns)

In [19]:
text_columns = ['title','description','requirements']
df['combined_text']=''
for col in text_columns:
    df['combined_text']= df['combined_text'] + ' ' + df[col].fillna('')




In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [21]:
tfidf = TfidfVectorizer(max_features=500,stop_words='english')
text_features = tfidf.fit_transform(df['combined_text'])

In [22]:
text_df = pd.DataFrame(text_features.toarray(),columns=tfidf.get_feature_names_out(),index=df.index)

In [23]:
df = pd.concat([df,text_df],axis=1)
df = df.drop(columns=text_columns + ['combined_text'])

In [ ]:
df.head()

: 

In [ ]:
df.columns


: 

In [24]:
from sklearn.model_selection import train_test_split
X = df.drop("fraudulent",axis=1)
y = df['fraudulent']

X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.2,random_state=42)


In [25]:
# X_train.dtypes

object_columns = X_train.select_dtypes(include=['object']).columns

X_train = X_train.drop(columns=object_columns)
X_test = X_test.drop(columns=object_columns)

In [26]:
X_train.dtypes.value_counts()

float64    525
int64        6
Name: count, dtype: int64

In [27]:
from sklearn.linear_model import  LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred=model.predict(X_test)


c:\adeelrana\envs\ml_algo\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [28]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


In [29]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1_value = f1_score(y_test, y_pred)
Confusion_M = confusion_matrix(y_test, y_pred)

print("Accuracy",accuracy)
print("precision",precision)
print("recall",recall)
print("f1_score",f1_value)
print("confusion",Confusion_M)


Accuracy 0.9584753641152773
precision 0.8571428571428571
recall 0.08333333333333333
f1_score 0.1518987341772152
confusion [[3081    2]
 [ 132   12]]


# PIPELINE IMPLEMENTATIION (NOT MANUALLY)

In [30]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import  LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder

In [34]:
categorical_columns = ['employment_type','required_experience','required_education']
text_columns = ['title','description','requirements']

text_transformer = TfidfVectorizer(max_features=500,stop_words="english")
categorical_transformer = OneHotEncoder(handle_unknown="ignore",sparse_output=False)

In [35]:
preprocessor = ColumnTransformer([
    ('text',text_transformer,'description'),
    ('cat',categorical_transformer,categorical_columns)
])

pipeline = Pipeline([
    ('preprocessing',preprocessor),
    ('model',LogisticRegression(max_iter=100))

])